This notebook conducts a series of experiments on the effect of using different lower bouds for fitting our exceedance function, while comparing them to a fit on the original Marani et al. data.

In [31]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from ipywidgets import interactive
from scipy.stats import genpareto

In [32]:
novel_outbreaks = pd.read_excel("../../data/raw/novel_resp_241228.xlsx")

*Run `python/scipts/fit_quasi_original_marani.py` before loading distribution configs below*

In [33]:
with open("../../output/severity_distributions/genpareto_quasi_original_marani_sev.yaml", 'rb') as f:
  orig_marani_sev_config = yaml.safe_load(f)
  
with open("../../output/severity_distributions/genpareto_quasi_original_marani_int.yaml", 'rb') as f:
  orig_marani_int_config = yaml.safe_load(f)
  
with open("../../data/clean/inverted_covid_severity.yaml", 'rb') as f:
  inverted_covid_severity_dict = yaml.safe_load(f)
  inverted_covid_severity = inverted_covid_severity_dict['ex_ante_severity']

In [39]:
def plot_exceedance(lower_bound=1e-2, drop_hiv=False, use_inverted_covid=False):
    """
    Creates interactive plot comparing exceedance functions between original Marani data and novel outbreaks
    
    Args:
        lower_bound (float): Lower bound threshold for fitting Pareto distribution
        drop_hiv (bool): Whether to exclude HIV/AIDS observation
        use_inverted_covid (bool): Whether to use inverted COVID-19 severity
    """
    # Create copy of novel outbreaks data
    df = novel_outbreaks.copy()
    
    # Apply HIV filter if selected
    if drop_hiv:
        df = df[df['disease'] != 'hiv/aids']
        
    # Replace COVID severity/intensity if selected
    if use_inverted_covid:
        covid_mask = df['disease'] == 'covid-19'
        df.loc[covid_mask, 'severity_smu'] = inverted_covid_severity
    
    # Filter by lower bound
    df_sev = df[df['severity_smu'] >= lower_bound]
    
    df['intensity'] = df['severity_smu'] / df['duration']
    df_int = df[df['intensity'] >= lower_bound]
    
    # Fit Pareto distributions
    sev_fit = genpareto.fit(df_sev['severity_smu'].values, floc=lower_bound)
    int_fit = genpareto.fit(df_int['intensity'].values, floc=lower_bound)
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15,6))
    
    # Generate points for theoretical curves
    x = np.logspace(np.log10(lower_bound), 3, 1000)
    
    # Plot original Marani fits
    y_sev_orig = 1 - genpareto.cdf(x, 
                                  c=orig_marani_sev_config['params']['k'],
                                  loc=orig_marani_sev_config['params']['theta'],
                                  scale=orig_marani_sev_config['params']['sigma'])
    
    y_int_orig = 1 - genpareto.cdf(x,
                                   c=orig_marani_int_config['params']['k'],
                                   loc=orig_marani_int_config['params']['theta'],
                                   scale=orig_marani_int_config['params']['sigma'])
    
    ax1.loglog(x, y_sev_orig, 'b-', label='~Original Marani Fit')
    ax2.loglog(x, y_int_orig, 'b-', label='~Original Marani Fit')
    
    # Plot new fits
    y_sev_new = 1 - genpareto.cdf(x, c=sev_fit[0], loc=sev_fit[1], scale=sev_fit[2])
    y_int_new = 1 - genpareto.cdf(x, c=int_fit[0], loc=int_fit[1], scale=int_fit[2])
    
    ax1.loglog(x, y_sev_new, 'r-', label='Novel Outbreaks Fit')
    ax2.loglog(x, y_int_new, 'r-', label='Novel Outbreaks Fit')
    
    # Plot empirical points and labels
    sev_sorted = np.sort(df_sev['severity_smu'].values)[::-1]
    int_sorted = np.sort(df_int['intensity'].values)[::-1]
    emp_exc_sev = np.arange(1, len(sev_sorted) + 1) / (len(sev_sorted) + 1)
    emp_exc_int = np.arange(1, len(int_sorted) + 1) / (len(int_sorted) + 1)
    
    ax1.loglog(sev_sorted, emp_exc_sev, 'ko', alpha=0.5)
    ax2.loglog(int_sorted, emp_exc_int, 'ko', alpha=0.5)
    
    # Add disease labels for severity plot
    for i, row in df_sev.iterrows():
        rank = (df_sev['severity_smu'] >= row['severity_smu']).sum()
        exc_prob = rank / (len(df_sev) + 1)
        ax1.annotate(row['disease'], (row['severity_smu'], exc_prob),
                    xytext=(5,5), textcoords='offset points', fontsize=8)
    
    # Add disease labels for intensity plot    
    for i, row in df_int.iterrows():
        rank = (df_int['intensity'] >= row['intensity']).sum()
        exc_prob = rank / (len(df_int) + 1)
        ax2.annotate(row['disease'], (row['intensity'], exc_prob),
                    xytext=(5,5), textcoords='offset points', fontsize=8)
    # Add vertical lines at lower bounds
    ax1.axvline(lower_bound, color='gray', linestyle='--', alpha=0.5)
    ax2.axvline(lower_bound*10, color='gray', linestyle='--', alpha=0.5)
    
    # Set labels and titles
    ax1.set_xlabel('Severity (deaths per 10k)')
    ax1.set_ylabel('Exceedance probability')
    ax1.set_title('Severity exceedance')
    ax1.legend()
    ax1.grid(True)
    
    ax2.set_xlabel('Intensity (deaths per 10k per year)')
    ax2.set_ylabel('Exceedance probability')
    ax2.set_title('Intensity exceedance')
    ax2.legend()
    ax2.grid(True)
    
    # Set consistent axis limits
    # ax1.set_ylim(min(emp_exc_sev), 1)
    # ax2.set_ylim(min(emp_exc_int), 1)
    
    plt.tight_layout()
    plt.show()

In [40]:
# Create interactive widget
interactive_plot = interactive(
    plot_exceedance,
    lower_bound=widgets.FloatLogSlider(value=1e-2, min=-6, max=1, step=0.1, description = 'Lower bound'),
    drop_hiv=widgets.Checkbox(value=False, description='Drop HIV/AIDS'),
    use_inverted_covid=widgets.Checkbox(value=False, description='Use Inverted COVID')
)

display(interactive_plot)


interactive(children=(FloatLogSlider(value=0.01, description='Lower bound', max=1.0, min=-6.0), Checkbox(value…